In [1]:
"""
NEW EXPERIMENTS — run this on your own GPU machine or Google Colab (needs a GPU runtime).
It will NOT run inside this chat (no GPU here, and the MedMNIST download servers are not
reachable from this sandbox). Copy this file into Colab, `pip install medmnist torch
torchvision`, then run section by section (each SECTION is written to be run independently).

Covers, in order:
  A. Shared utilities (data loading, equal-budget epoch scaling)
  B. Mixture-of-Experts federated model (FedMoE-style) — the new 7th method
  C. Centralized vs Federated with a strictly equal training budget (S vs S/n)
  D. Full 6-method bootstrap campaign on BloodMNIST (the new-disease task)
  E. Diagnostic script to find the exact cause of a 0% bootstrap result
  F. Hyperparameters / workflow, written as a machine-readable dict so it can be pasted
     straight into your Methods section

Everything here plugs into your existing training loop (FedAvg/FedProx/FedAdam/
FederatedDistillation/Centralized/Local-only) — it assumes those four federated
trainers already exist in your codebase (federated_medical_multiclass_v2.ipynb) and
imports/reuses them rather than re-implementing FedAvg etc. Adjust the import line
in SECTION A to match your actual notebook's module name if you convert it to a .py file.
"""

import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

## SECTION A — shared utilities

In [2]:
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Your five original clients + the sixth "new disease" client used for bootstrapping.
CLIENTS = ["pneumonia", "breast", "derma", "path", "retina"]
BOOTSTRAP_CLIENT = "blood"  # BloodMNIST — never seen during main federated training

def make_resnet18_backbone(pretrained=True, freeze_first_layer=True):
    """Same backbone described in your report: ImageNet ResNet18, only conv1 frozen,
    BatchNorm kept LOCAL/personal per client (do not FedAvg the BN buffers/affine params)."""
    import torchvision.models as models
    net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    if freeze_first_layer:
        for p in net.conv1.parameters():
            p.requires_grad = False
    return net

def bn_param_names(model):
    """Names of BatchNorm parameters/buffers that must stay LOCAL (never averaged)."""
    names = []
    for n, m in model.named_modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            names += [f"{n}.weight", f"{n}.bias", f"{n}.running_mean", f"{n}.running_var"]
    return set(names)

def equal_budget_epochs(centralized_total_samples, n_clients, fed_local_epochs, fed_rounds):
    """
    Point 4 — make Centralized and Federated see the SAME total number of
    gradient-update samples, not just the same epoch count.

    Federated: each of n_clients sees S/n samples per local epoch, for
    `fed_local_epochs` local epochs per round, across `fed_rounds` rounds.
        total_samples_seen_fed = n_clients * (S/n) * fed_local_epochs * fed_rounds
                                = S * fed_local_epochs * fed_rounds

    So to match Centralized exactly:
        centralized_epochs = fed_local_epochs * fed_rounds

    This is the single-line fix for point 4 — Centralized should NOT simply run for
    the same number of "epochs" as one federated client's local epoch count; it must
    run for (local_epochs * rounds) epochs to have seen the same total volume of data.
    """
    centralized_epochs = fed_local_epochs * fed_rounds
    return centralized_epochs

## SECTION B — Mixture-of-Experts Federated Learning (new 7th method)

In [3]:
# Rationale (already supported by your own data): CKA similarity between clients'
# learned features was only ~0.268 (Report Section 9.2) — the five modalities do NOT
# converge on one shared representation. That is exactly the condition MoE is built
# for: keep a shared trunk small, and let a gate route each image to a
# modality-specialized expert head instead of forcing one shared head to fit all five.
#
# This implements a FedMoE-style model:
#   - one shared ResNet18 trunk (federated-averaged, excluding BatchNorm as before)
#   - K expert heads (K = number of clients, or fewer with soft sharing)
#   - a small gating network trained per-client that learns to weight experts
# This is the standard "sparse mixture of experts" pattern (Shazeer et al. 2017,
# adapted to federated learning as FedMoE / FedMix-style routing). Cite the specific
# papers you use once you've run this — see literature note at bottom of this file.

class GatingNetwork(nn.Module):
    def __init__(self, feat_dim, n_experts):
        super().__init__()
        self.fc = nn.Linear(feat_dim, n_experts)

    def forward(self, feats):
        return F.softmax(self.fc(feats), dim=-1)  # [B, n_experts]

class MoEHead(nn.Module):
    def __init__(self, feat_dim, n_experts, n_classes_per_expert):
        """n_classes_per_expert: dict {expert_idx: num_classes} since your clients
        have different category counts (2..9). Each expert has its own output size."""
        super().__init__()
        self.n_experts = n_experts
        self.experts = nn.ModuleList([
            nn.Linear(feat_dim, n_classes_per_expert[i]) for i in range(n_experts)
        ])
        self.gate = GatingNetwork(feat_dim, n_experts)

    def forward(self, feats, expert_hint=None):
        """expert_hint: during federated training on client i, you already know which
        client you're on, so you can supervise the gate to prefer expert i (auxiliary
        loss) while still letting it blend in others (soft routing)."""
        gate_weights = self.gate(feats)  # [B, n_experts]
        if expert_hint is not None:
            # hard-route during training on a known client, use only that expert's head
            return self.experts[expert_hint](feats), gate_weights
        # soft blend at inference (e.g. for the 25-category multidisease test):
        # pad each expert's logits to the full 25-way space before blending
        return None, gate_weights  # fill in per-expert logits + padding at call site

class FedMoEModel(nn.Module):
    def __init__(self, n_clients, classes_per_client, pretrained=True):
        super().__init__()
        self.trunk = make_resnet18_backbone(pretrained=pretrained)
        feat_dim = self.trunk.fc.in_features
        self.trunk.fc = nn.Identity()
        self.head = MoEHead(feat_dim, n_clients, classes_per_client)

    def forward(self, x, client_idx=None):
        feats = self.trunk(x)
        logits, gate_w = self.head(feats, expert_hint=client_idx)
        return logits, gate_w

def moe_aux_gate_loss(gate_weights, true_client_idx):
    """Auxiliary loss: encourage (not force) the gate to prefer the correct expert,
    while still allowing soft blending — standard load-balanced MoE trick, simplified."""
    target = torch.full((gate_weights.size(0),), true_client_idx, dtype=torch.long, device=gate_weights.device)
    return F.cross_entropy(gate_weights, target)

# Training loop sketch — plug into your existing federated round structure:
def train_fedmoe_round(model, client_loaders, optimizer_per_client, local_epochs=1, aux_weight=0.1):
    """One FedMoE round. Only `model.trunk` parameters get FedAvg'd across clients
    afterward (same BN-exclusion rule as your other methods); each client's own
    expert head stays local/personalized — that IS the point of MoE, so do not
    average the expert heads themselves, only the shared trunk + gate."""
    client_states = {}
    for client_idx, loader in enumerate(client_loaders):
        local_model = copy.deepcopy(model).to(DEVICE)
        opt = optimizer_per_client[client_idx]
        local_model.train()
        for _ in range(local_epochs):
            for x, y in loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                opt.zero_grad()
                logits, gate_w = local_model(x, client_idx=client_idx)
                loss = F.cross_entropy(logits, y) + aux_weight * moe_aux_gate_loss(gate_w, client_idx)
                loss.backward()
                opt.step()
        client_states[client_idx] = local_model.state_dict()
    # FedAvg only the trunk + gate, excluding BN and excluding expert head weights
    bn_names = bn_param_names(model.trunk)
    expert_head_names = {n for n in model.state_dict() if "head.experts" in n}
    exclude = bn_names | expert_head_names
    avg_state = {}
    for k in model.state_dict():
        if k in exclude:
            continue  # leave as-is / handled per-client
        avg_state[k] = torch.stack([client_states[i][k].float() for i in range(len(client_loaders))]).mean(0)
    model.load_state_dict(avg_state, strict=False)
    return model, client_states

## SECTION C — Equal-budget Centralized vs Federated campaign (point 4)

In [4]:
def run_equal_budget_campaign(fed_local_epochs=1, fed_rounds=8, n_clients=5):
    """
    This is the fix for point 4. Re-run your EXISTING Centralized trainer (no new
    model needed) but with:
        centralized_epochs = equal_budget_epochs(S, n_clients, fed_local_epochs, fed_rounds)
    instead of whatever epoch count you used before, so both sides see the same
    TOTAL number of (sample, gradient-step) pairs, only distributed differently
    (S in one place vs S/n per client x n clients).

    Report BOTH of these two controls explicitly in your Methods section, since they
    answer slightly different questions:
      (1) equal total samples seen  -> centralized_epochs = fed_local_epochs * fed_rounds
      (2) equal wall-clock/communication budget -> centralized_epochs = fed_rounds
    Run both and put them side by side in the new table; that comparison itself
    ("does the FL vs Centralized gap survive a fair compute budget?") is a
    legitimate, citable methodological contribution on its own.
    """
    centralized_epochs_equal_samples = equal_budget_epochs(None, n_clients, fed_local_epochs, fed_rounds)
    centralized_epochs_equal_rounds = fed_rounds
    print(f"Run Centralized with epochs={centralized_epochs_equal_samples} (equal total samples seen)")
    print(f"Run Centralized with epochs={centralized_epochs_equal_rounds} (equal communication budget)")
    # >>> call your existing train_centralized(epochs=...) here for each of the two values <<<

## SECTION D — Full bootstrap campaign, all 6 methods, on BloodMNIST (point 2)

In [5]:
METHODS = ["Centralized", "Local_only", "FedAvg", "FedProx", "FedAdam", "FederatedDistillation"]
DATA_FRACTIONS = [0.10, 0.25, 0.50, 1.00]

def run_full_bootstrap_campaign(pretrained_backbones):
    """
    pretrained_backbones: dict mapping each of the 6 method names above to the
    already-trained backbone checkpoint from your MAIN 5-client experiment
    (Section 3-6 of your report). For each method's backbone, fine-tune a fresh
    classification head on BloodMNIST at each data fraction and evaluate — exactly
    mirroring the structure of your existing Table 5, but for ALL SIX methods
    instead of just {from-scratch, ImageNet, federated}.

    This directly answers point 2: "compare centralized vs different HFL and
    repeat all the same steps/campaign of experiments" for the bootstrap task.
    """
    results = []
    for method in METHODS:
        backbone_ckpt = pretrained_backbones[method]
        for frac in DATA_FRACTIONS:
            # >>> replace with your real BloodMNIST loader + your existing
            #     fine-tune-and-eval function; keep epochs/lr identical across
            #     methods so the comparison stays fair <<<
            # acc, f1, auc = finetune_and_eval(backbone_ckpt, "blood", frac)
            # results.append({"method": method, "fraction": frac, "accuracy": acc, "f1": f1, "auc": auc})
            pass
    return results

## SECTION E — Diagnostic for a 0% bootstrap result (point 3)

In [6]:
def diagnose_zero_percent(model, loader, expected_n_classes, class_names=None):
    """
    Run this BEFORE re-training anything. A clean 0% (not ~random-guess-level, but
    a hard zero) on a real dataset is almost always one of these four bugs, checked
    here in order of likelihood:
    """
    model.eval()
    report = {}

    # 1. Output layer / class-head mismatch — most common cause.
    out_dim = None
    for m in reversed(list(model.modules())):
        if isinstance(m, nn.Linear):
            out_dim = m.out_features
            break
    report["output_layer_size"] = out_dim
    report["expected_n_classes"] = expected_n_classes
    report["HEAD_MISMATCH"] = (out_dim != expected_n_classes)

    # 2. Label mapping mismatch — dataset labels not in [0, n_classes-1], or an old
    #    25-category combined label being fed into a model expecting 0..5.
    all_labels = []
    with torch.no_grad():
        for _, y in loader:
            all_labels.append(y)
    all_labels = torch.cat(all_labels)
    report["label_min"] = int(all_labels.min())
    report["label_max"] = int(all_labels.max())
    report["LABEL_RANGE_BUG"] = (all_labels.min() < 0 or all_labels.max() >= expected_n_classes)

    # 3. Frozen/never-updated head — check the head actually has requires_grad=True
    #    and that at least one optimizer step touched it.
    head_frozen = []
    for n, p in model.named_parameters():
        if "fc" in n or "head" in n:
            head_frozen.append((n, p.requires_grad))
    report["head_params_trainable"] = head_frozen

    # 4. Prediction collapse — model always predicts the same single class
    #    (classic sign of a stale BatchNorm running_mean/var from a different
    #    client/modality being reused without recalibration).
    preds = []
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(DEVICE)
            logits = model(x)
            if isinstance(logits, tuple):
                logits = logits[0]
            preds.append(logits.argmax(1).cpu())
    preds = torch.cat(preds)
    unique_preds = torch.unique(preds)
    report["unique_predictions"] = unique_preds.tolist()
    report["COLLAPSED_TO_SINGLE_CLASS"] = (len(unique_preds) == 1)

    print(json.dumps({k: v for k, v in report.items() if k != "head_params_trainable"}, indent=2))
    for n, trainable in head_frozen:
        if not trainable:
            print(f"WARNING: head param '{n}' has requires_grad=False — it never trained.")
    return report

## SECTION F — Experimental parameters / workflow (paste into Methods section)

In [7]:
EXPERIMENT_CONFIG = {
    "backbone": "ResNet18, ImageNet-pretrained, conv1 frozen, all other layers fine-tuned",
    "batch_normalization": "kept LOCAL/personal per client — never included in FedAvg/FedProx/FedAdam aggregation",
    "image_size": "96x96",
    "communication_rounds": 8,
    "local_epochs_per_round": "<<< FILL IN from your notebook >>>",
    "batch_size": "<<< FILL IN >>>",
    "optimizer": "<<< FILL IN, e.g. Adam >>>",
    "learning_rate": "<<< FILL IN >>>",
    "fedprox_mu": "<<< FILL IN >>>",
    "fedadam_server_lr_beta1_beta2": "<<< FILL IN >>>",
    "distillation_shared_set_size": "250 (see Table 10 in report)",
    "runs_per_result": "1 (single run — repeat 3-5x with different seeds if time allows, report mean+-std)",
    "new_equal_budget_rule": "centralized_epochs = fed_local_epochs * fed_rounds (see SECTION C)",
    "new_moe_method": "FedMoE-style: shared trunk + gate FedAvg'd (excl. BN), expert heads kept local per client",
    "bootstrap_dataset": "BloodMNIST — 8 classes, never used in main 5-client training",
}

if __name__ == "__main__":
    set_seed()
    print("This file defines experiment code — run individual sections in Colab/GPU.")
    print(json.dumps(EXPERIMENT_CONFIG, indent=2))

This file defines experiment code — run individual sections in Colab/GPU.
{
  "backbone": "ResNet18, ImageNet-pretrained, conv1 frozen, all other layers fine-tuned",
  "batch_normalization": "kept LOCAL/personal per client \u2014 never included in FedAvg/FedProx/FedAdam aggregation",
  "image_size": "96x96",
  "communication_rounds": 8,
  "local_epochs_per_round": "<<< FILL IN from your notebook >>>",
  "batch_size": "<<< FILL IN >>>",
  "optimizer": "<<< FILL IN, e.g. Adam >>>",
  "learning_rate": "<<< FILL IN >>>",
  "fedprox_mu": "<<< FILL IN >>>",
  "fedadam_server_lr_beta1_beta2": "<<< FILL IN >>>",
  "distillation_shared_set_size": "250 (see Table 10 in report)",
  "runs_per_result": "1 (single run \u2014 repeat 3-5x with different seeds if time allows, report mean+-std)",
  "new_equal_budget_rule": "centralized_epochs = fed_local_epochs * fed_rounds (see SECTION C)",
  "new_moe_method": "FedMoE-style: shared trunk + gate FedAvg'd (excl. BN), expert heads kept local per client",
